<a href="https://colab.research.google.com/github/GraceMolina/SQL-Problem-Solving/blob/main/AvailabilityAtOrderTime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Filtering data from tables: orders, line_items, items, item_field_history, and events
"""

In [ ]:
# mount google drive to read the dataset
from google.colab import drive
drive.mount('/content/drive')

# read the dataset
import pandas as pd

file_path = '/content/drive/MyDrive/SQL Problem Solving/Module 1/Dataset.xlsx'

data = pd.ExcelFile(file_path)
print(data.sheet_names)

# read sheets
sheets = pd.read_excel(file_path, sheet_name = None)

for name, df in sheets.items():
  print(name, df.shape)
  display(df.head())



In [ ]:
# read the sheets into a SQL database
import sqlite3

conn = sqlite3.connect(':memory:')

for name, df in sheets.items():
  table = name.strip().replace(' ', '_')
  df.to_sql(table, conn, index = False, if_exists = 'replace')

# confirm the tables exist
pd.read_sql('SELECT name FROM sqlite_master WHERE type = "table"', conn)

In [ ]:
# count data recorded for the event 'item_view'
query1 = """
SELECT COUNT(id) AS id_count,
       COUNT(user_id) AS user_id_count,
       COUNT(session_id) AS session_id_count,
       COUNT(device_id) AS device_id_count,
       COUNT(event_name) AS event_name_count,
       COUNT(user_properties) AS user_properties_count,
       COUNT(event_properties) AS event_properties_count,
       COUNT(event_time) AS event_time_count,
       COUNT(client_event_time) AS client_event_time_count,
       COUNT(server_upload_time) AS server_upload_time_count
FROM events
WHERE EVENT_NAME = 'item_view'
"""
result1 = pd.read_sql(query1, conn)
result1

In [ ]:
# count data recorded for the event 'view_item'
query2 = """
SELECT COUNT(id) AS id_count,
       COUNT(user_id) AS user_id_count,
       COUNT(session_id) AS session_id_count,
       COUNT(device_id) AS device_id_count,
       COUNT(event_name) AS event_name_count,
       COUNT(user_properties) AS user_properties_count,
       COUNT(event_properties) AS event_properties_count,
       COUNT(event_time) AS event_time_count,
       COUNT(client_event_time) AS client_event_time_count,
       COUNT(server_upload_time) AS server_upload_time_count
FROM events
WHERE EVENT_NAME = 'view_item'
"""
result2 = pd.read_sql(query2, conn)
result2

In [ ]:
# look at availability status at the time an item was ordered
query3 = """
WITH availability_changes AS (
    SELECT id,
           record_id,
           changed_timestamp,
           previous_value,
           new_value
    FROM item_field_history
    WHERE field_name = 'availability'
),
order_lines AS (
    SELECT li.id AS line_item_id,
           li.order_id,
           li.item_id,
           o.created_timestamp
    FROM line_items li
    LEFT JOIN orders o
    ON li.order_id = o.id
),
last_change_before AS (
SELECT ol.line_item_id,
       ac.new_value,
       ROW_NUMBER() OVER (
        PARTITION BY ol.line_item_id
        ORDER BY ac.changed_timestamp DESC, ac.id DESC
        ) AS rn
FROM order_lines ol
JOIN availability_changes ac
ON ol.item_id = ac.record_id AND ol.created_timestamp >= ac.changed_timestamp
),
first_change_after AS (
SELECT ol.line_item_id,
       ac.previous_value,
       ROW_NUMBER() OVER (
        PARTITION BY ol.line_item_id
        ORDER BY ac.changed_timestamp ASC, ac.id ASC
        ) AS rn
FROM order_lines ol
JOIN availability_changes ac
ON ol.item_id = ac.record_id AND ol.created_timestamp < ac.changed_timestamp
)
SELECT ol.line_item_id AS id,
       ol.order_id,
       ol.item_id,
       ol.created_timestamp,
       CASE
        WHEN lb.line_item_id IS NOT NULL THEN lb.new_value
        WHEN fa.line_item_id IS NOT NULL THEN fa.previous_value
        ELSE i.availability
       END AS availability_when_ordered
FROM order_lines ol
LEFT JOIN last_change_before lb
ON ol.line_item_id = lb.line_item_id AND lb.rn = 1
LEFT JOIN first_change_after fa
ON ol.line_item_id = fa.line_item_id AND fa.rn = 1
LEFT JOIN items i
ON ol.item_id = i.id
"""
result3 = pd.read_sql(query3, conn)
result3

In [ ]:
# extract item_id from the event_properties in the events table
# approach 1
query4 = """
SELECT event_properties->>'$.item_id' AS item_id
FROM events
"""
result4 = pd.read_sql(query4, conn)
result4

In [ ]:
# approach 2
query5 = """
SELECT
JSON_EXTRACT(event_properties, '$.item_id') AS item_id
FROM events
"""
result5 = pd.read_sql(query5, conn)
result5